# InduSense — Sprint 1 : cartes d'identité des fichiers CSV

Pour chaque fichier CSV de `datas/`, ce notebook produit une carte d'identité Markdown dans
`artifacts/ingestion/identities/<AAAAMMJJ>/<nom_fichier>_<HHMMSS>.md` contenant :

- la source, la taille, l'heure de début et de fin de traitement, le charset,
- le nombre de lignes et de colonnes,
- la liste des colonnes avec leur type inféré,
- min / max (et mean / median pour les numériques) des colonnes numériques et dates.

In [1]:
import csv
from datetime import datetime
from decimal import Decimal
from pathlib import Path

import pandas as pd
import sqlglot
from IPython.display import Markdown, display
from sqlglot import exp

DATA_DIR = Path("datas")
ARTIFACTS_DIR = Path("artifacts/ingestion/identities")
CSV_FILES = [
    DATA_DIR / "releves_incidents.csv",
    DATA_DIR / "telemetry.csv",
]
SQL_FILE = DATA_DIR / "machine.sql.txt"
SQL_DIALECT = "postgres"

## 1. Lecture du fichier : taille, charset et séparateur

Le charset est déterminé en essayant de décoder le fichier (ASCII, puis UTF-8, avec ou sans BOM).
À défaut, on considère le fichier en ISO-8859-1 (Latin-1), qui accepte n'importe quel octet.

Le séparateur n'est pas toujours une virgule : il est détecté par `csv.Sniffer` parmi `,` `;` `\t` `|`.

In [2]:
def format_filesize(size_bytes: int) -> str:
    """Formate une taille en octets : 9999.99 [octets | Ko | Mo | Go]."""
    size = float(size_bytes)
    for unit in ("octets", "Ko", "Mo"):
        if size < 1024:
            return f"{size:.2f} {unit}"
        size /= 1024
    return f"{size:.2f} Go"


def detect_charset(path: Path) -> tuple[str, str]:
    """Retourne (libellé du charset, encodage Python à utiliser pour lire le fichier)."""
    raw = path.read_bytes()
    if raw.startswith(b"\xef\xbb\xbf"):
        return "UTF-8 (BOM)", "utf-8-sig"
    for label, encoding in (("ASCII", "ascii"), ("UTF-8", "utf-8")):
        try:
            raw.decode(encoding)
            return label, encoding
        except UnicodeDecodeError:
            pass
    return "ISO-8859-1 (Latin-1)", "latin-1"


def detect_separator(path: Path, encoding: str) -> str:
    """Détecte le séparateur du CSV à partir d'un échantillon du début du fichier."""
    with path.open(encoding=encoding, newline="") as f:
        sample = f.read(64 * 1024)
    try:
        return csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
    except csv.Error:
        return ","


for path in CSV_FILES:
    charset, encoding = detect_charset(path)
    print(f"{path.name}: {format_filesize(path.stat().st_size)}, {charset}, "
          f"séparateur {detect_separator(path, encoding)!r}")

releves_incidents.csv: 134.64 Ko, UTF-8, séparateur ','
telemetry.csv: 7.94 Mo, ASCII, séparateur ','


## 2. Inférence du type des colonnes

| Type | Règle |
| --- | --- |
| `IDENTITY` | nom `id` / `*_id` / `*_code`, ou valeurs de la forme `MACH-01`, `INC-000001`, `OP1002` (au moins 2 chiffres) |
| `BOOLEAN` | valeurs uniquement 0 / 1 (ou booléens) |
| `INTEGER` / `FLOAT` | colonne numérique |
| `DATE` / `DATETIME` | valeurs au format ISO 8601 (`2025-06-01`, `2025-06-01 05:42:00`) |
| `TIME` | valeurs `HH:MM` ou `HH:MM:SS` |
| `CATEGORY` | texte avec au plus 20 valeurs distinctes |
| `TEXT` | autre texte |

In [3]:
IDENTITY_PATTERN = r"[A-Za-z]+-?\d{2,}"
IDENTITY_SUFFIXES = ("_id", "_code")
TIME_PATTERN = r"\d{2}:\d{2}(:\d{2})?"
MAX_CATEGORIES = 20

NUMERIC_TYPES = {"INTEGER", "FLOAT"}
TEMPORAL_TYPES = {"DATE", "DATETIME", "TIME"}


def infer_column_type(name: str, series: pd.Series) -> str:
    """Infère le type métier d'une colonne."""
    values = series.dropna()
    if values.empty:
        return "EMPTY"
    if name.lower() == "id" or name.lower().endswith(IDENTITY_SUFFIXES):
        return "IDENTITY"
    if pd.api.types.is_bool_dtype(values):
        return "BOOLEAN"
    if pd.api.types.is_numeric_dtype(values):
        if set(values.unique()) <= {0, 1}:
            return "BOOLEAN"
        return "INTEGER" if pd.api.types.is_integer_dtype(values) else "FLOAT"

    text = values.astype(str).str.strip()
    if text.str.fullmatch(IDENTITY_PATTERN).all():
        return "IDENTITY"
    if text.str.fullmatch(TIME_PATTERN).all():
        return "TIME"
    dates = pd.to_datetime(text, format="ISO8601", errors="coerce")
    if dates.notna().all():
        is_date_only = (dates == dates.dt.normalize()).all() and text.str.len().max() <= 10
        return "DATE" if is_date_only else "DATETIME"
    if text.nunique() <= MAX_CATEGORIES:
        return "CATEGORY"
    return "TEXT"

## 3. Statistiques des colonnes numériques et dates

Min et max pour toutes ces colonnes ; mean et median uniquement pour les numériques.

In [4]:
def format_value(value) -> str:
    """Formate une valeur pour l'affichage dans la carte d'identité."""
    if value is None:
        return "-"
    if isinstance(value, float):
        return f"{value:.3f}"
    return str(value)


def column_stats(series: pd.Series, col_type: str) -> dict:
    """Calcule min / max (et mean / median pour les numériques) d'une colonne."""
    values = series.dropna()
    if col_type in NUMERIC_TYPES:
        return {
            "min": values.min().item(),
            "max": values.max().item(),
            "mean": float(values.mean()),
            "median": float(values.median()),
        }
    if col_type == "TIME":
        # Format HH:MM zéro-paddé : l'ordre alphabétique est l'ordre chronologique
        text = values.astype(str).str.strip()
        return {"min": text.min(), "max": text.max(), "mean": None, "median": None}
    dates = pd.to_datetime(values.astype(str).str.strip(), format="ISO8601")
    date_format = "%Y-%m-%d" if col_type == "DATE" else "%Y-%m-%d %H:%M:%S"
    return {
        "min": dates.min().strftime(date_format),
        "max": dates.max().strftime(date_format),
        "mean": None,
        "median": None,
    }

## 4. Construction et écriture de la carte d'identité

In [5]:
def build_identity(path: Path) -> dict:
    """Analyse un fichier CSV et retourne sa carte d'identité sous forme de dictionnaire."""
    begin = datetime.now()
    charset, encoding = detect_charset(path)
    separator = detect_separator(path, encoding)
    df = pd.read_csv(path, sep=separator, encoding=encoding)

    types = {col: infer_column_type(col, df[col]) for col in df.columns}
    stats = {
        col: column_stats(df[col], col_type)
        for col, col_type in types.items()
        if col_type in NUMERIC_TYPES | TEMPORAL_TYPES
    }
    return {
        "source": path.resolve(),
        "filesize": format_filesize(path.stat().st_size),
        "begin": begin,
        "end": datetime.now(),
        "charset": charset,
        "rows": len(df),
        "cols": len(df.columns),
        "types": types,
        "stats": stats,
    }


def render_file_header(identity: dict) -> list[str]:
    """Lignes Markdown communes à toutes les cartes : source, taille, horaires, charset."""
    return [
        f"# Carte d'identité : {identity['source'].name}",
        "",
        f"- Source: {identity['source']}",
        f"- Filesize: {identity['filesize']}",
        f"- ProcessBeginAt: {identity['begin']:%H:%M:%S}",
        f"- ProcessEndAt: {identity['end']:%H:%M:%S}",
        f"- Charset: {identity['charset']}",
    ]


def render_stats_table(stats: dict) -> list[str]:
    """Tableau Markdown des statistiques des colonnes numériques et dates."""
    lines = [
        "| Colname | Min value | Max value | Mean value | Median value |",
        "| --- | --- | --- | --- | --- |",
    ]
    lines += [
        f"| {col} | {format_value(s['min'])} | {format_value(s['max'])} "
        f"| {format_value(s['mean'])} | {format_value(s['median'])} |"
        for col, s in stats.items()
    ]
    return lines


def render_identity(identity: dict) -> str:
    """Met en forme la carte d'identité d'un CSV en Markdown."""
    lines = render_file_header(identity) + [
        f"- Rows Number: {identity['rows']}",
        f"- Cols Number: {identity['cols']}",
        "",
        "## Column list",
        "",
        "| Colname | Inferred Type |",
        "| --- | --- |",
    ]
    lines += [f"| {col} | {col_type} |" for col, col_type in identity["types"].items()]
    lines += ["", "## Numeric and date columns", ""] + render_stats_table(identity["stats"])
    return "\n".join(lines) + "\n"


def write_identity(identity: dict, render=render_identity, output_root: Path = ARTIFACTS_DIR) -> Path:
    """Écrit la carte dans <output_root>/<AAAAMMJJ>/<nom_fichier>_<HHMMSS>.md."""
    begin = identity["begin"]
    output_dir = output_root / f"{begin:%Y%m%d}"
    output_dir.mkdir(parents=True, exist_ok=True)
    file_name = identity["source"].name.split(".")[0]  # machine.sql.txt -> machine
    output_path = output_dir / f"{file_name}_{begin:%H%M%S}.md"
    output_path.write_text(render(identity), encoding="utf-8")
    return output_path

## 5. Génération des cartes d'identité

In [6]:
for path in CSV_FILES:
    identity = build_identity(path)
    output_path = write_identity(identity)
    print(f"Carte générée : {output_path}")
    display(Markdown(render_identity(identity)))

Carte générée : artifacts/ingestion/identities/20261120/releves_incidents_145652.md


# Carte d'identité : releves_incidents.csv

- Source: /home/thopuetai/Formation IA/indusense/datas/releves_incidents.csv
- Filesize: 134.64 Ko
- ProcessBeginAt: 14:56:52
- ProcessEndAt: 14:56:52
- Charset: UTF-8
- Rows Number: 1245
- Cols Number: 18

## Column list

| Colname | Inferred Type |
| --- | --- |
| incident_id | IDENTITY |
| date | DATE |
| time | TIME |
| operator_name | CATEGORY |
| machine_id | IDENTITY |
| severity | INTEGER |
| operator_badge | IDENTITY |
| comment | TEXT |
| shift | CATEGORY |
| type_surchauffe | BOOLEAN |
| type_baisse_pression | BOOLEAN |
| type_vibration | BOOLEAN |
| type_bruit_mecanique | BOOLEAN |
| type_surconsommation | BOOLEAN |
| type_blocage_mecanique | BOOLEAN |
| type_alarme_capteur | BOOLEAN |
| type_arret_urgence | BOOLEAN |
| type_defaut_qualite | BOOLEAN |

## Numeric and date columns

| Colname | Min value | Max value | Mean value | Median value |
| --- | --- | --- | --- | --- |
| date | 2025-06-01 | 2026-06-08 | - | - |
| time | 00:06 | 23:55 | - | - |
| severity | 2 | 5 | 2.925 | 3.000 |


Carte générée : artifacts/ingestion/identities/20261120/telemetry_145652.md


# Carte d'identité : telemetry.csv

- Source: /home/thopuetai/Formation IA/indusense/datas/telemetry.csv
- Filesize: 7.94 Mo
- ProcessBeginAt: 14:56:52
- ProcessEndAt: 14:56:53
- Charset: ASCII
- Rows Number: 135626
- Cols Number: 7

## Column list

| Colname | Inferred Type |
| --- | --- |
| machine_id | IDENTITY |
| timestamp | DATETIME |
| temperature_c | FLOAT |
| pressure_bar | FLOAT |
| voltage_mean_v | FLOAT |
| rotation_mean_rpm | FLOAT |
| pieces_produced | INTEGER |

## Numeric and date columns

| Colname | Min value | Max value | Mean value | Median value |
| --- | --- | --- | --- | --- |
| timestamp | 2025-06-01 00:00:00 | 2026-06-08 23:00:00 | - | - |
| temperature_c | 32.000 | 80.000 | 48.183 | 48.055 |
| pressure_bar | 159.982 | 215.814 | 199.770 | 199.866 |
| voltage_mean_v | 220.900 | 242.000 | 227.631 | 227.420 |
| rotation_mean_rpm | 1100.000 | 1900.000 | 1589.205 | 1590.370 |
| pieces_produced | 0 | 114 | 49.533 | 49.000 |


## 6. Pour aller plus loin : carte d'identité du script SQL

`machine.sql.txt` est un script PostgreSQL : il ne se lit pas comme un tableau. Il faut un parseur SQL
capable d'en extraire deux informations :

- `CREATE TABLE` donne les colonnes et leur **type SQL déclaré**,
- `INSERT INTO ... VALUES` donne les **données**, chargées dans un DataFrame par table.

### Choix de la librairie : sqlglot

| Option | Ce qu'elle fournit | Pourquoi elle n'a pas été retenue |
| --- | --- | --- |
| **[sqlglot](https://github.com/tobymao/sqlglot)** ✅ | Arbre syntaxique complet (AST) : tables, colonnes, types, valeurs déjà typées (`int`, `Decimal`, `None`...) | — |
| [sqlparse](https://github.com/andialbrecht/sqlparse) | Découpage en instructions et en *tokens* (parseur « non validant ») | Pas d'arbre syntaxique : il faudrait reconstruire à la main colonnes, types et lignes de `VALUES` en parcourant les tokens |
| [pglast](https://github.com/lelit/pglast) | Le vrai parseur de PostgreSQL (libpg_query) | Extension C : demande des binaires compilés pour notre version de Python (3.15) ; limitée au seul dialecte PostgreSQL |
| `sqlite3` (bibliothèque standard) | Exécute le script dans une base en mémoire, puis on relit les tables | Ce n'est pas un parseur : il faut que SQLite accepte le script, or celui-ci utilise des syntaxes propres à PostgreSQL (`DEFAULT NOW()`, `TIMESTAMPTZ`...) ; et on perd les types déclarés |

sqlglot coche toutes les cases pour ce besoin :

- **Python pur, sans aucune dépendance** : `uv add sqlglot` n'installe qu'un seul paquet, rien à compiler,
- **comprend le dialecte PostgreSQL** (`read="postgres"`) : `TIMESTAMPTZ`, `NUMERIC(6,2)`, `ON CONFLICT ... DO UPDATE`...,
- **aucune base de données nécessaire** : le script est analysé, jamais exécuté,
- **arbre syntaxique navigable** : `exp.Create`, `exp.Insert`, `exp.ColumnDef`... rendent l'extraction directe,
- projet très utilisé et activement maintenu.

Chaque table est ensuite traitée comme un CSV, avec les mêmes fonctions `infer_column_type` et `column_stats`.
Les colonnes déclarées mais absentes des `INSERT` (remplies par une valeur par défaut, ex. `created_at`)
apparaissent avec le type inféré `-`.

```shell
uv add sqlglot
```

In [7]:
def literal_value(node: exp.Expression):
    """Convertit une valeur SQL (texte, nombre, NULL...) en valeur Python."""
    value = node.to_py()
    return float(value) if isinstance(value, Decimal) else value


def extract_sql_tables(statements: list[exp.Expression]) -> dict:
    """Pour chaque table : types déclarés (CREATE TABLE) et données insérées (INSERT ... VALUES)."""
    tables = {}

    def get_table(name: str) -> dict:
        return tables.setdefault(name, {"sql_types": {}, "frames": []})

    for statement in statements:
        if isinstance(statement, exp.Create) and statement.kind == "TABLE":
            schema = statement.this
            get_table(schema.this.name)["sql_types"] = {
                col.name: col.args["kind"].sql(SQL_DIALECT)
                for col in schema.expressions
                if isinstance(col, exp.ColumnDef)
            }
        elif isinstance(statement, exp.Insert) and isinstance(statement.expression, exp.Values):
            target = statement.this
            if isinstance(target, exp.Schema):  # INSERT INTO table (col1, col2, ...)
                table = get_table(target.this.name)
                columns = [col.name for col in target.expressions]
            else:  # INSERT INTO table VALUES ... : toutes les colonnes déclarées
                table = get_table(target.name)
                columns = list(table["sql_types"])
            rows = [[literal_value(v) for v in row.expressions] for row in statement.expression.expressions]
            table["frames"].append(pd.DataFrame(rows, columns=columns))

    return {
        name: {
            "sql_types": table["sql_types"],
            "data": pd.concat(table["frames"], ignore_index=True) if table["frames"] else pd.DataFrame(),
        }
        for name, table in tables.items()
    }

In [8]:
def build_sql_identity(path: Path) -> dict:
    """Analyse un script SQL et retourne sa carte d'identité (une section par table)."""
    begin = datetime.now()
    charset, encoding = detect_charset(path)
    statements = sqlglot.parse(path.read_text(encoding=encoding), read=SQL_DIALECT)

    tables = {}
    for name, table in extract_sql_tables(statements).items():
        data = table["data"]
        types = {col: infer_column_type(col, data[col]) for col in data.columns}
        tables[name] = {
            "rows": len(data),
            "sql_types": table["sql_types"],
            "types": types,
            "stats": {
                col: column_stats(data[col], col_type)
                for col, col_type in types.items()
                if col_type in NUMERIC_TYPES | TEMPORAL_TYPES
            },
        }
    return {
        "source": path.resolve(),
        "filesize": format_filesize(path.stat().st_size),
        "begin": begin,
        "end": datetime.now(),
        "charset": charset,
        "dialect": SQL_DIALECT,
        "statements": len(statements),
        "tables": tables,
    }


def render_sql_identity(identity: dict) -> str:
    """Met en forme la carte d'identité d'un script SQL en Markdown."""
    lines = render_file_header(identity) + [
        f"- SQL Dialect: {identity['dialect']}",
        f"- Statements Number: {identity['statements']}",
        f"- Tables Number: {len(identity['tables'])}",
    ]
    for name, table in identity["tables"].items():
        columns = list(table["sql_types"]) + [c for c in table["types"] if c not in table["sql_types"]]
        lines += [
            "",
            f"## Table `{name}`",
            "",
            f"- Rows Number: {table['rows']}",
            f"- Cols Number: {len(columns)}",
            "",
            "### Column list",
            "",
            "| Colname | SQL Type | Inferred Type |",
            "| --- | --- | --- |",
        ]
        lines += [
            f"| {col} | {table['sql_types'].get(col, '-')} | {table['types'].get(col, '-')} |"
            for col in columns
        ]
        lines += ["", "### Numeric and date columns", ""] + render_stats_table(table["stats"])
    return "\n".join(lines) + "\n"

In [9]:
identity = build_sql_identity(SQL_FILE)
output_path = write_identity(identity, render=render_sql_identity)
print(f"Carte générée : {output_path}")
display(Markdown(render_sql_identity(identity)))

Carte générée : artifacts/ingestion/identities/20261120/machine_145653.md


# Carte d'identité : machine.sql.txt

- Source: /home/thopuetai/Formation IA/indusense/datas/machine.sql.txt
- Filesize: 22.41 Ko
- ProcessBeginAt: 14:56:53
- ProcessEndAt: 14:56:53
- Charset: UTF-8
- SQL Dialect: postgres
- Statements Number: 10
- Tables Number: 2

## Table `machine`

- Rows Number: 15
- Cols Number: 11

### Column list

| Colname | SQL Type | Inferred Type |
| --- | --- | --- |
| machine_code | VARCHAR(16) | IDENTITY |
| commissioning_date | DATE | DATE |
| max_daily_capacity | INT | INTEGER |
| max_hourly_capacity_pieces | INT | INTEGER |
| model | VARCHAR(32) | CATEGORY |
| production_line | VARCHAR(16) | CATEGORY |
| location | VARCHAR(16) | CATEGORY |
| criticality | VARCHAR(8) | CATEGORY |
| is_active | BOOLEAN | - |
| created_at | TIMESTAMPTZ | - |
| updated_at | TIMESTAMPTZ | - |

### Numeric and date columns

| Colname | Min value | Max value | Mean value | Median value |
| --- | --- | --- | --- | --- |
| commissioning_date | 2019-07-23 | 2025-05-25 | - | - |
| max_daily_capacity | 750 | 1428 | 1054.867 | 1027.000 |
| max_hourly_capacity_pieces | 47 | 89 | 65.867 | 64.000 |

## Table `maintenance`

- Rows Number: 115
- Cols Number: 11

### Column list

| Colname | SQL Type | Inferred Type |
| --- | --- | --- |
| maintenance_id | INT | IDENTITY |
| machine_code | VARCHAR(16) | IDENTITY |
| maintenance_at | TIMESTAMPTZ | DATETIME |
| maintenance_type | VARCHAR(16) | CATEGORY |
| action_type | VARCHAR(32) | CATEGORY |
| component | VARCHAR(64) | CATEGORY |
| description | TEXT | TEXT |
| related_incident_id | VARCHAR(16) | IDENTITY |
| duration_hours | DECIMAL(6, 2) | FLOAT |
| created_at | TIMESTAMPTZ | - |
| updated_at | TIMESTAMPTZ | - |

### Numeric and date columns

| Colname | Min value | Max value | Mean value | Median value |
| --- | --- | --- | --- | --- |
| maintenance_at | 2025-06-15 06:00:00 | 2026-05-30 22:00:00 | - | - |
| duration_hours | 1.000 | 8.000 | 2.943 | 2.640 |
